# Image Feature Classifier v1.0

`eda v2`에서 만든 CV/preprocessing feature만 사용해 tabular classifier를 학습하고 test inference까지 수행합니다.

In [8]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

import lightgbm as lgb
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

ROOT = Path.cwd().resolve().parent
DATA_DIR = ROOT / "data"
FEATURE_CSV = ROOT / "outputs" / "eda_cv" / "cv_features_max150.csv"
SUBMISSION_DIR = ROOT / "outputs" / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

assert FEATURE_CSV.exists(), f"feature csv not found: {FEATURE_CSV}"
print("FEATURE_CSV:", FEATURE_CSV)
            


FEATURE_CSV: /media/hdd0/whyz/structure-stability/outputs/eda_cv/cv_features_max150.csv


In [9]:
def build_sample_level_features(feat_df: pd.DataFrame) -> pd.DataFrame:
    meta_cols = ["split", "sample_id", "label"]
    value_cols = [c for c in feat_df.columns if c not in meta_cols + ["view", "image_path"]]

    wide_parts = []
    for view in sorted(feat_df["view"].unique()):
        part = feat_df.loc[feat_df["view"] == view, meta_cols + value_cols].copy()
        part = part.rename(columns={col: f"{view}_{col}" for col in value_cols})
        wide_parts.append(part)

    sample_df = wide_parts[0]
    for part in wide_parts[1:]:
        sample_df = sample_df.merge(part, on=meta_cols, how="outer")

    common_value_cols = [
        col
        for col in value_cols
        if f"front_{col}" in sample_df.columns and f"top_{col}" in sample_df.columns
    ]
    for col in common_value_cols:
        sample_df[f"delta_{col}"] = sample_df[f"front_{col}"] - sample_df[f"top_{col}"]

    return sample_df


feat_df = pd.read_csv(FEATURE_CSV)
feat_df = feat_df.loc[feat_df["split"] != "generated_v2"].copy()
sample_df = build_sample_level_features(feat_df)
sample_df["target"] = sample_df["label"].map({"stable": 0, "unstable": 1})

print("sample_df shape:", sample_df.shape)
display(sample_df.head())
display(sample_df.groupby("split").size().rename("rows").reset_index())
            


sample_df shape: (400, 109)


,split,sample_id,label,front_width,front_height,front_mean_r,front_mean_g,front_mean_b,front_std_r,front_std_g,...,delta_vp_pitch_proxy,delta_vp_intersections,delta_distortion_proxy,delta_shadow_area_ratio,delta_shadow_angle_deg,delta_shadow_elongation,delta_chessboard_found,delta_chessboard_nx,delta_chessboard_ny,target
0,dev,DEV_001,unstable,384,384,172.413947,181.888041,198.582933,31.124773,24.013015,...,-0.103266,235,0.048669,NaN,NaN,NaN,0,0,0,1.0
1,dev,DEV_002,unstable,384,384,185.392524,193.651903,211.024957,42.192023,40.175989,...,-0.109693,125,0.055043,NaN,NaN,NaN,0,0,0,1.0
2,dev,DEV_003,unstable,384,384,192.676439,204.514086,223.686727,44.590903,40.118734,...,-0.139944,342,0.059978,NaN,NaN,NaN,0,0,0,1.0
3,dev,DEV_004,stable,384,384,180.458469,189.944071,204.682997,45.687536,41.826257,...,-0.088053,169,0.037338,NaN,NaN,NaN,1,6,6,0.0
4,dev,DEV_005,stable,384,384,175.368069,185.876044,201.347188,38.773011,32.660092,...,-0.094648,250,0.048444,NaN,NaN,NaN,0,0,0,0.0


,split,rows
0,dev,100
1,test,150
2,train,150


In [10]:
train_df = sample_df.loc[sample_df["split"] == "train"].copy().reset_index(drop=True)
dev_df = sample_df.loc[sample_df["split"] == "dev"].copy().reset_index(drop=True)
test_df = sample_df.loc[sample_df["split"] == "test"].copy().reset_index(drop=True)

feature_cols = [c for c in sample_df.columns if c not in ["split", "sample_id", "label", "target"]]
feature_cols = [c for c in feature_cols if sample_df[c].notna().any()]

imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(imputer.fit_transform(train_df[feature_cols]), columns=feature_cols)
X_dev = pd.DataFrame(imputer.transform(dev_df[feature_cols]), columns=feature_cols)
X_test = pd.DataFrame(imputer.transform(test_df[feature_cols]), columns=feature_cols)
y_train = train_df["target"].to_numpy()
y_dev = dev_df["target"].to_numpy()

print("n_features:", len(feature_cols))
print("train/dev/test:", X_train.shape, X_dev.shape, X_test.shape)
            


n_features: 99
train/dev/test: (150, 99) (100, 99) (150, 99)


In [11]:
SEED = 42
EARLY_STOPPING_ROUNDS = 50

models = {
    "xgboost": XGBClassifier(
        n_estimators=700,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1,
    ),
    "lightgbm": LGBMClassifier(
        n_estimators=700,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.8,
        random_state=SEED,
    ),
    "catboost": CatBoostClassifier(
        iterations=700,
        learning_rate=0.03,
        depth=6,
        loss_function="Logloss",
        eval_metric="Logloss",
        random_seed=SEED,
        verbose=False,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        random_state=SEED,
        n_jobs=-1,
    ),
    "extra_trees": ExtraTreesClassifier(
        n_estimators=700,
        min_samples_leaf=2,
        random_state=SEED,
        n_jobs=-1,
    ),
}
            


In [12]:
eval_rows = []
dev_pred_map = {}
refit_test_pred_map = {}
feature_importance_rows = []
test_sample_ids = test_df["sample_id"].to_numpy()


def fit_with_dev_early_stopping(name, model, X_train, y_train, X_dev, y_dev):
    train_model = clone(model)

    if name == "xgboost":
        train_model.set_params(early_stopping_rounds=EARLY_STOPPING_ROUNDS)
        train_model.fit(X_train, y_train, eval_set=[(X_dev, y_dev)], verbose=False)
        best_rounds = int(train_model.best_iteration + 1)
    elif name == "lightgbm":
        train_model.fit(
            X_train,
            y_train,
            eval_set=[(X_dev, y_dev)],
            eval_metric="binary_logloss",
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )
        best_rounds = int(train_model.best_iteration_)
    elif name == "catboost":
        train_model.fit(
            X_train,
            y_train,
            eval_set=(X_dev, y_dev),
            use_best_model=True,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose=False,
        )
        best_rounds = int(train_model.get_best_iteration() + 1)
    else:
        train_model.fit(X_train, y_train)
        best_rounds = None

    return train_model, best_rounds


def fit_refit_model(name, model, X_full, y_full, best_rounds):
    refit_model = clone(model)

    if best_rounds is not None:
        if name == "xgboost":
            refit_model.set_params(n_estimators=best_rounds)
        elif name == "lightgbm":
            refit_model.set_params(n_estimators=best_rounds)
        elif name == "catboost":
            refit_model.set_params(iterations=best_rounds)

    refit_model.fit(X_full, y_full)
    return refit_model

for name, model in models.items():
    train_model, best_rounds = fit_with_dev_early_stopping(name, model, X_train, y_train, X_dev, y_dev)
    dev_pred = train_model.predict_proba(X_dev)[:, 1]

    eval_rows.append({
        "model": name,
        "dev_logloss": float(log_loss(y_dev, dev_pred)),
        "dev_auc": float(roc_auc_score(y_dev, dev_pred)),
        "dev_acc": float(accuracy_score(y_dev, (dev_pred >= 0.5).astype(int))),
        "best_rounds": best_rounds,
    })
    dev_pred_map[name] = dev_pred

    if hasattr(train_model, "feature_importances_"):
        importances = pd.Series(train_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
        for feature, importance in importances.head(20).items():
            feature_importance_rows.append({"model": name, "feature": feature, "importance": float(importance)})

    X_full = pd.concat([X_train, X_dev], axis=0, ignore_index=True)
    y_full = np.concatenate([y_train, y_dev])
    refit_model = fit_refit_model(name, model, X_full, y_full, best_rounds)
    refit_test_pred_map[name] = refit_model.predict_proba(X_test)[:, 1]

eval_df = pd.DataFrame(eval_rows).sort_values(["dev_logloss", "dev_auc"], ascending=[True, False]).reset_index(drop=True)
display(eval_df)

if feature_importance_rows:
    importance_df = pd.DataFrame(feature_importance_rows)
    display(importance_df.sort_values(["model", "importance"], ascending=[True, False]).groupby("model").head(10))
            


[LightGBM] [Info] Number of positive: 80, number of negative: 70
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000463 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3798
[LightGBM] [Info] Number of data points in the train set: 150, number of used features: 89
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.533333 -> initscore=0.133531
[LightGBM] [Info] Start training from score 0.133531
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

,model,dev_logloss,dev_auc,dev_acc,best_rounds
0,catboost,0.667625,0.690104,0.57,10.0
1,xgboost,0.681847,0.681090,0.57,6.0
2,lightgbm,0.684661,0.618790,0.58,5.0
3,extra_trees,0.964051,0.756410,0.52,NaN
4,random_forest,0.989889,0.786058,0.52,NaN


,model,feature,importance
40,catboost,top_structure_center_x,51.293764
41,catboost,front_structure_center_x,23.255017
42,catboost,delta_structure_bbox_w,4.180640
43,catboost,delta_fft_high_freq_ratio,2.918428
44,catboost,top_structure_bbox_w,2.545244
45,catboost,front_shadow_area_ratio,2.545048
46,catboost,delta_structure_center_x,1.788799
47,catboost,top_mean_g,1.209341
48,catboost,front_mean_r,1.153415
49,catboost,delta_noise_residual_std,1.138497


In [13]:
top_k = min(3, len(eval_df))
top_models = eval_df.head(top_k)["model"].tolist()
weights = 1.0 / eval_df.head(top_k)["dev_logloss"].to_numpy()
weights = weights / weights.sum()

print("top_models:", top_models)
print("weights:", dict(zip(top_models, weights.round(4))))

dev_blend = np.average(np.column_stack([dev_pred_map[m] for m in top_models]), axis=1, weights=weights)
test_blend = np.average(np.column_stack([refit_test_pred_map[m] for m in top_models]), axis=1, weights=weights)

blend_row = {
    "model": "weighted_topk_ensemble",
    "dev_logloss": float(log_loss(y_dev, dev_blend)),
    "dev_auc": float(roc_auc_score(y_dev, dev_blend)),
    "dev_acc": float(accuracy_score(y_dev, (dev_blend >= 0.5).astype(int))),
}
display(pd.concat([eval_df, pd.DataFrame([blend_row])], ignore_index=True))
            


top_models: ['catboost', 'xgboost', 'lightgbm']
weights: {'catboost': np.float64(0.3385), 'xgboost': np.float64(0.3314), 'lightgbm': np.float64(0.3301)}


,model,dev_logloss,dev_auc,dev_acc,best_rounds
0,catboost,0.667625,0.690104,0.57,10.0
1,xgboost,0.681847,0.681090,0.57,6.0
2,lightgbm,0.684661,0.618790,0.58,5.0
3,extra_trees,0.964051,0.756410,0.52,NaN
4,random_forest,0.989889,0.786058,0.52,NaN
5,weighted_topk_ensemble,0.676495,0.703125,0.58,NaN


In [14]:
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")

for name, probs in refit_test_pred_map.items():
    pred_df = pd.DataFrame({
        "sample_id": test_sample_ids,
        "unstable_prob": probs,
        "stable_prob": 1.0 - probs,
    })
    pred_path = SUBMISSION_DIR / f"image_feature_classifier_{name}_v1.0_test_subset.csv"
    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
    print("saved subset predictions:", pred_path)

    merged_sub = sample_submission.merge(pred_df, left_on="id", right_on="sample_id", how="left")
    matched = int(merged_sub["unstable_prob_y"].notna().sum())
    if matched == len(sample_submission):
        sub = sample_submission.copy()
        sub["unstable_prob"] = merged_sub["unstable_prob_y"].to_numpy()
        sub["stable_prob"] = merged_sub["stable_prob_y"].to_numpy()
        out_path = SUBMISSION_DIR / f"image_feature_classifier_{name}_v1.0.csv"
        sub.to_csv(out_path, index=False, encoding="utf-8-sig")
        print("saved official submission:", out_path)
    else:
        print(
            f"skipped official submission for {name}: "
            f"matched {matched}/{len(sample_submission)} sample ids from feature csv"
        )

ensemble_pred_df = pd.DataFrame({
    "sample_id": test_sample_ids,
    "unstable_prob": test_blend,
    "stable_prob": 1.0 - test_blend,
})
ensemble_pred_path = SUBMISSION_DIR / "image_feature_classifier_ensemble_v1.0_test_subset.csv"
ensemble_pred_df.to_csv(ensemble_pred_path, index=False, encoding="utf-8-sig")
print("saved subset predictions:", ensemble_pred_path)

ensemble_merged = sample_submission.merge(ensemble_pred_df, left_on="id", right_on="sample_id", how="left")
ensemble_matched = int(ensemble_merged["unstable_prob_y"].notna().sum())
if ensemble_matched == len(sample_submission):
    ensemble_sub = sample_submission.copy()
    ensemble_sub["unstable_prob"] = ensemble_merged["unstable_prob_y"].to_numpy()
    ensemble_sub["stable_prob"] = ensemble_merged["stable_prob_y"].to_numpy()
    ensemble_out = SUBMISSION_DIR / "image_feature_classifier_ensemble_v1.0.csv"
    ensemble_sub.to_csv(ensemble_out, index=False, encoding="utf-8-sig")
    print("saved official submission:", ensemble_out)
else:
    print(
        "skipped official ensemble submission: "
        f"matched {ensemble_matched}/{len(sample_submission)} sample ids from feature csv"
    )

summary_path = ROOT / "outputs" / "eda_preprocessing" / "image_feature_classifier_v1.0_eval.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
pd.concat([eval_df, pd.DataFrame([blend_row])], ignore_index=True).to_csv(summary_path, index=False)
print("saved:", summary_path)
            


saved subset predictions: /media/hdd0/whyz/structure-stability/outputs/submissions/image_feature_classifier_xgboost_v1.0_test_subset.csv
skipped official submission for xgboost: matched 150/1000 sample ids from feature csv
saved subset predictions: /media/hdd0/whyz/structure-stability/outputs/submissions/image_feature_classifier_lightgbm_v1.0_test_subset.csv
skipped official submission for lightgbm: matched 150/1000 sample ids from feature csv
saved subset predictions: /media/hdd0/whyz/structure-stability/outputs/submissions/image_feature_classifier_catboost_v1.0_test_subset.csv
skipped official submission for catboost: matched 150/1000 sample ids from feature csv
saved subset predictions: /media/hdd0/whyz/structure-stability/outputs/submissions/image_feature_classifier_random_forest_v1.0_test_subset.csv
skipped official submission for random_forest: matched 150/1000 sample ids from feature csv
saved subset predictions: /media/hdd0/whyz/structure-stability/outputs/submissions/image_fea